### This notebook contains my final model attempt of getting the highest accuracy using retreival augmented fact checking (scraping the internet for facts)

##### I tried many different methods with BERT to try and train it and use it to classify misinformation however I hit a wall and could not improve the model any further. I know am going to attempt to make a retreival augmented model by utilizing wikipedia and DeBERTa-v3-base

In [ ]:
%pip install transformers accelerate datasets sentencepiece scikit-learn pandas numpy wikipedia tqdm evaluate matplotlib

In [ ]:
import torch
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
import subprocess
subprocess.run(["nvidia-smi"])

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
df = pd.read_csv("combinedClaimDataset.csv")
df.head()

In [ ]:
TARGET_FEVER_SIZE = 40000

fever_df = df[df["source"] == "fever"]
liar_df = df[df["source"] == "liar"]
scifact_df = df[df["source"] == "scifact"]

if len(fever_df) > TARGET_FEVER_SIZE:
    fever_df_small = (
        fever_df
        .groupby("label", group_keys=False)
        .apply(lambda x: x.sample(
            frac=min(1.0, TARGET_FEVER_SIZE / len(fever_df)),
            random_state=42
        ))
        .reset_index(drop=True)
    )
else:
    fever_df_small = fever_df.copy()

df_balanced = pd.concat([fever_df_small, liar_df, scifact_df], ignore_index=True)
df_balanced = df_balanced.sample(frac=1.0, random_state=42).reset_index(drop=True)

df_balanced["source"].value_counts()

In [ ]:
train_df, val_df = train_test_split(
    df_balanced,
    test_size=0.3,
    random_state=42,
    stratify=df_balanced["label"]
)
len(train_df), len(val_df)

In [ ]:
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["claim"],
        batch["evidence_text"],
        truncation=True,
        padding="max_length",
        max_length=256  # optimized for speed
    )

In [ ]:
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    label_smoothing_factor=0.05
).to(device)

In [ ]:
training_args = TrainingArguments(
    output_dir="./model_output",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,      # 1 epoch = safe under 1 hour
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
preds = trainer.predict(val_ds)
logits = preds.predictions
labels = preds.label_ids

probs = torch.softmax(torch.tensor(logits), dim=1).numpy()

best_thresh = 0
best_f1 = 0

for t in np.linspace(0.1, 0.9, 81):
    pred_labels = (probs[:,1] > t).astype(int)
    f1 = f1_score(labels, pred_labels, average="macro")
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

best_thresh, best_f1